# Imports and data:

In [1]:
import pandas as pd
import numpy as np
import regex as re

In [2]:
w1 = pd.read_excel("data/WINNERS 2000-2010 - 05.xlsx")
w2 = pd.read_excel("data/WINNERS 2011-2020 - 05.xlsx")
w3 = pd.read_excel("data/WINNERS 2021-2025 - 05.xlsx")
w4 = pd.read_excel("data/WINNERS 2026-2030 - 05.xlsx")

In [3]:
winners = pd.concat([w1, w2, w3, w4], ignore_index=True)

winners.shape

(43457, 10)

In [4]:
winners.dtypes

Name                   object
School                 object
Year                    int64
City                   object
Award                  object
Division               object
Category               object
School / Company       object
Division / Position    object
Dance Category         object
dtype: object

In [5]:
winners

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
0,NaN,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
1,NaN,"The Kirov Academy, DC",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
2,8+3,Inwood Dance Company,2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
3,Cabaret,"Dance Conservatory, DE",2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
4,Make Them Hear You,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",2nd Place,NaN,Ensembles,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
43452,Dance For Six,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43453,Coppelia Friends Dance,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43454,Kira Fargas,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN
43455,Hadassah Perry,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN


In [6]:
winners.isna().sum()

Name                     554
School                 36132
Year                       0
City                       0
Award                    174
Division               36913
Category               36477
School / Company        8286
Division / Position    11327
Dance Category         10780
dtype: int64

- Some awards are for the school overall rather than a person
- Some awards are for the name of the dance rather than people
- School awards wouldn't have a Dance Category or Position, which would explain some entries being null
- NaNs in certain columns can actually tell you what kind of award is being given:
    - NaN in School/Company ; School Award
- Some people don't belong in a School/Company

# Data Cleanup:

- Whenever Name and School have NaNs, there are no rows where School/Company is empty
    - So: replace School with whatever is in the School/Company for that row

In [7]:
winners[
    (winners.Name.isna()) & 
    (winners.School.isna())
]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
7717,NaN,NaN,2011,"Torrington, CT",Outstanding School Award,NaN,NaN,"Washington School of Ballet, DC, USA",NaN,Special Awards
7806,NaN,NaN,2011,"Tampa, FL",Outstanding School Award,NaN,NaN,The Art of Classical Ballet,NaN,Special Awards
7807,NaN,NaN,2011,"Tampa, FL",Outstanding School Award,NaN,NaN,America's Ballet School,NaN,Special Awards
7905,NaN,NaN,2011,"San Francisco, CA",Outstanding School Award,NaN,NaN,"Los Angeles Ballet Academy, CA",NaN,Special Awards
7906,NaN,NaN,2011,"San Francisco, CA",Outstanding School Award,NaN,NaN,"Westlake School for the Performing Arts, CA",NaN,Special Awards
...,...,...,...,...,...,...,...,...,...,...
42644,NaN,NaN,2026,"Boca Raton, FL",Outstanding School Award,NaN,NaN,"Stars Dance Studio, FL",Special Awards,NaN
42928,NaN,NaN,2026,"Indianapolis, IN",Outstanding School Award,NaN,NaN,"Indiana Ballet Conservatory, IN",Special Awards,NaN
43200,NaN,NaN,2026,"Boston-Worcester, MA",Outstanding School Award,NaN,NaN,"N&D Ballet, MA",Special Awards,NaN
43201,NaN,NaN,2026,"Boston-Worcester, MA",Outstanding School Award,NaN,NaN,"Koltun Ballet Boston, MA",Special Awards,NaN


In [8]:
winners[
    (winners.Name.isna()) & 
    (winners.School.isna()) &
    (winners["School / Company"].isna())
]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category


In [9]:
mask = (winners["Name"].isna()) & (winners["School"].isna())

winners.loc[mask, "School"] = winners.loc[mask, "School / Company"]
winners

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
0,NaN,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
1,NaN,"The Kirov Academy, DC",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
2,8+3,Inwood Dance Company,2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
3,Cabaret,"Dance Conservatory, DE",2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
4,Make Them Hear You,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",2nd Place,NaN,Ensembles,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
43452,Dance For Six,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43453,Coppelia Friends Dance,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43454,Kira Fargas,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN
43455,Hadassah Perry,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN


In [10]:
mask =  (winners.School.isna()) & (~winners["School / Company"].isna())
winners.loc[
    mask,
    "School"    
] = winners.loc[mask, "School / Company"]

- Inconsistent patterns for recording names:
    - First Name Last Name(s)
    - Last Name, First Name
    - First Name Last Name, [US state] (sometimes followed by what US state they're from) 
    - First Name Middle Name(s) and Last Name(s)
    - [1st person] & [2nd person]
    - [1st person], [2nd person]
    - [1st person] (age), [2nd person] (age)
    - (age)
    - [1st person] age, [2nd person] age
    - First Name Last Name (age), (dance that the person/people performed)
    - dance that the person/people performed, ([1st person], [2nd person]), 
- Removing hanging whitespace from all string columns since it will effect analysis
- Turn everything uppercase, since there are inconsistent capitalization (or lack thereof)

In [11]:
winners["Name"] = winners.Name.str.strip()

In [12]:
winners = winners.apply(lambda col: col.str.upper() if col.dtype == "object" else col)

In [13]:
names = sorted(winners.Name.dropna().unique())

with open("data/yagp_names.txt", "w") as file:
    file.writelines(
        name + "\n" for name in names
    )

In [14]:
def split_2_names(string):
    if pd.isna(string):
        return string  # keep NaN
    
    if isinstance(string, str):
        return [s.strip() for s in string.split("&")]
    
    return string

winners.Name = winners.Name.apply(split_2_names)
winners = winners.explode("Name", ignore_index=True)

In [15]:
def normalize_spaces(s):
    return re.sub(r"\s+", " ", str(s).strip())

# all non-comma names to use as reference
reference_names = set(
    normalize_spaces(x)
    for x in winners["Name"].dropna()
    if isinstance(x, str) and "," not in x
)

def resolve_comma_name(name):
    if not isinstance(name, str) or "," not in name:
        return name, "not_comma"

    parts = [p.strip() for p in name.split(",", 1)]
    if len(parts) != 2 or not parts[0] or not parts[1]:
        return name, "bad_format"

    left, right = parts[0], parts[1]

    candidate_flip = f"{right} {left}"
    candidate_keep = f"{left} {right}"

    flip_exists = candidate_flip in reference_names
    keep_exists = candidate_keep in reference_names

    if flip_exists and not keep_exists:
        return candidate_flip, "confirmed_reversed"
    elif keep_exists and not flip_exists:
        return candidate_keep, "already_correct_or_unclear"
    elif flip_exists and keep_exists:
        return candidate_flip, "both_exist_ambiguous"
    else:
        return name, "ambiguous"

In [16]:
# ----------------------------
# name/person patterns
# ----------------------------

# standard person name
name_std = re.compile(
    r"^[\p{Lu}][\p{L}\p{M}'’´`.\-]*(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){1,6}$"
)

# reversed name: LAST, FIRST ...
name_rev = re.compile(
    r"^[\p{Lu}][\p{L}\p{M}'’´`.\-]+,\s*[\p{Lu}][\p{L}\p{M}'’´`.\-]*(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){0,5}$"
)

# name followed by state
name_state = re.compile(
    r"^[\p{Lu}][\p{L}\p{M}'’´`.\-]*(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){1,6},\s*[A-Z]{2}$"
)

# person name with nickname / alternate name in parentheses
# examples: JUNZIONG (JAKE) ZHAO, MIMI (FIONA CLAIRE) CAMPBELL
name_nickname = re.compile(
    r"^[\p{Lu}][\p{L}\p{M}'’´`.\-]*"
    r"(?:\s+\([\p{Lu}\p{L}\p{M}'’´`.\-\s]+\))"
    r"(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){1,6}$"
)

# person name with native-script version appended
# examples: YOONJI LEE 이윤지
name_native_script = re.compile(
    r"^[\p{Lu}][\p{L}\p{M}'’´`.\-]*(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){1,6}\s+[\p{L}\p{M}]+$"
)

# multiple people
two_and = re.compile(
    r"^[\p{Lu}][\p{L}\p{M}'’´`.\-]*(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){1,6}"
    r"\s+(?:AND|&)\s+"
    r"[\p{Lu}][\p{L}\p{M}'’´`.\-]*(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){1,6}$"
)

two_age_paren = re.compile(
    r"^[\p{Lu}][\p{L}\p{M}'’´`.\-]*(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){1,6}\s*\(\d{1,2}\),\s*"
    r"[\p{Lu}][\p{L}\p{M}'’´`.\-]*(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){1,6}\s*\(\d{1,2}\)$"
)

two_age_plain = re.compile(
    r"^[\p{Lu}][\p{L}\p{M}'’´`.\-]*(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){1,6},\s*\d{1,2},\s*"
    r"[\p{Lu}][\p{L}\p{M}'’´`.\-]*(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){1,6},\s*\d{1,2}$"
)

single_age = re.compile(r"^\(\d{1,2}\)$")

# person with age + dance note
name_age_dance = re.compile(
    r"^[\p{Lu}][\p{L}\p{M}'’´`.\-]*(?:\s+[\p{Lu}][\p{L}\p{M}'’´`.\-]*){1,6}\s*\(\d{1,2}\)(?:,\s*|\s+)\([^)]*\)?$"
)

# merged names with no spaces: ALANAGRIFFITH, MARIARODRIGUEZ
merged_name_like = re.compile(
    r"^[\p{Lu}][\p{L}\p{M}'’´`\-]{7,}$"
)

# ----------------------------
# school patterns
# ----------------------------
school_pat = re.compile(
    r"\b(?:ACADEMY|SCHOOL|STUDIO|STUDIOS|CONSERVATORY|DANCENTER|COMPANY|THEATRE|THEATER|BALLET)\b"
)

# ----------------------------
# dataset-specific title words
# ----------------------------
title_tokens = {
    "A", "AN", "THE", "AND", "OF", "IN", "ON", "TO", "FROM", "WITH", "FOR",
    "YOU", "YOUR", "MY", "ME", "OUR", "US", "THEM", "IT",
    "PAS", "DE", "DEUX", "TROIS", "TRIO", "VARIATION", "VARIATIONS",
    "SCENE", "ACT", "SUITE", "PRELUDE", "WALTZ", "MAZURKA", "FUGUE",
    "FESTIVAL", "FLOWER", "FLOWERS", "PEARLS", "DOLLS", "FRIENDS",
    "NUTCRACKER", "GISELLE", "COPPELIA", "PAQUITA", "BAYADERE",
    "CORSAIRE", "HARLEQUINADE", "SATANELLA", "ESMERALDA", "SYLPHIDE",
    "TARANTELLA", "TARANTELLE", "CARMEN", "ODALISQUES", "SWAN", "LAKE",
    "WORLD", "VOICE", "LOVE", "HOLD", "HEAR", "LOUD", "RAIN", "DREAM",
    "ILLUSION", "VISION", "RHYTHM", "HOME", "HORIZONS", "STORM",
    "REFUGE", "MIRROR", "RIBBON", "HONEY", "BEES", "NOCTURNE", "SPANISH",
    "RUSSIAN", "MOLDAVIAN", "DANCE", "SONG", "BALL", "HEART", "EYES",
    "FREEDOM", "ENCOUNTER", "LIBERTANGO", "REQUIEM", "BERCEUSE", "INTRIGUE",
    "CAPTIVE", "CHOPINIANA", "MOZART", "CINDERELLA", "PAQUITA", "FUGACE"
}

single_word_non_names = {
    "CABARET", "CAPRICE", "RONDO", "BALLETSAAL", "ILLUSION", "VISION",
    "HOME", "HOMETOWN", "HOMEWARD", "HORIZONS", "HOSPITAL", "HUMAN",
    "HUMANITY", "HYPNOSIS", "PACE", "PALLADIO", "PARTITA", "PARADIGM",
    "PARADOX", "PARALLELS", "ASCENDING", "ASCENDURA", "ASPIRATIONS",
    "ATMOS", "APOCALYSIS", "APPASSIONATA", "APPASSIONATO", "ALLEGRO",
    "ADAGIO", "ADORATION", "AMORE", "ALLIANCE", "AURORA", "BANYAN",
    "BEAT", "BEAUTIFUL", "BLOOMING", "BLUE", "SEEK", "FROST", "FANTASY",
    "INVITATION", "KALEIDOSCOPE", "PULSE", "SYMBION", "TEMPESTUOUS",
    "CARMEN", "SPANISH", "LIMP", "MERLITON", "ODALISQUES", "TARANTELLE",
    "NAVARRA", "ENCUENTRO", "VOLARE", "FLOCK", "PRALAYA", "TRIAD",
    "KURAYAMINO", "MERLITONS", "NOCTURNE", "STORM", "ESPANOL", "PRESSAGE",
    "BEES", "EXOTICA", "UNTITLED", "HOEDOWN", "OKLAHOMA", "JESTERS",
    "ALLEGRETTO", "VIVIFIED", "RESURRECTION", "GORECKI", "VIGIL",
    "VOLCANO", "MERCY", "GRAY", "GOPAK", "TRIO", "POLKA", "SUMMERTIME",
    "GREY", "PRECE", "LAGAAN", "SANKANANDA", "MOZARTIA", "UNDERGROUND",
    "LEYANDA", "OOPS", "TWINS", "VIVALDI", "ENCOUNTER", "WINNER",
    "FUGACE", "PAQUITA", "CINDERELLA", "MANCHEGA", "KOKYU", "RECIFERVENDO",
    "TRINARY", "INTERIORS", "SOMEDAY", "RECESS", "SWING", "LIBERTANGO",
    "CIRANDA", "MUJIERES", "ANDANTE", "REQUIEM", "PAGANISSIMO", "CAPTIVE",
    "CHOPINIANA", "BERCEUSE", "REMINISCENCE", "INTRIGUE", "KODO",
    "AERODYNAMIC", "BANDONEON", "BOSSACUCANOVA", "BOLERO", "MÓBILE",
    "OFF", "PLENTITUDO", "SATANELLA", "CANCAN", "DOLLS", "CHERRY",
    "INANE", "MOVIES", "FREETIME", "PURSUIT", "SCHERZO", "SENIORITAS",
    "SOJOURNER", "TCHOKOLO", "MOZART", "APOLOGIZE", "FREEDOM",
    "COWGIRLS", "MENDELSSOHN", "AISURU", "IMPLOSION", "MARKITENKA",
    "STARFIGHTER", "ENSALADAS", "FLY", "TECHNO", "RIDE", "ENERGY", "USO",
    "IMPROMPTU", "CLOCKS", "TRAPPED", "SHADOW", "BEATBOX", "JOTA",
    "ROADBLOCK", "VALKYRIES", "ANUGRAHA"
}

def normalize_line(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def looks_like_title_not_name(s: str) -> bool:
    upper = normalize_line(s).upper()

    # obvious punctuation / title structure
    if re.search(r'[#"/?]|^\d|^\(|\)$', upper):
        return True

    # school/company
    if school_pat.search(upper):
        return True

    # one-word ambiguous entries that are not names in this dataset
    if " " not in upper and upper in single_word_non_names:
        return True

    tokens = [t for t in re.split(r"\s+", upper) if t]
    cleaned_tokens = [re.sub(r"^[^\p{L}]+|[^\p{L}]+$", "", t) for t in tokens]
    cleaned_tokens = [t for t in cleaned_tokens if t]

    bad_count = sum(t in title_tokens for t in cleaned_tokens)

    # if it contains multiple title words, it's probably a title
    if bad_count >= 2:
        return True

    # classic title patterns from this dataset
    if re.search(
        r"\b(?:PAS DE DEUX|PAS DE TROIS|FLOWER FESTIVAL|FROM NUTCRACKER|FROM GISELLE|FROM COPPELIA|FROM DON QUIXOTE)\b",
        upper
    ):
        return True

    # title-like phrases
    if len(cleaned_tokens) >= 3 and bad_count >= 1:
        return True

    return False

def maybe_manual_review_name(s: str) -> bool:
    """
    Catch probable names that lost spaces:
    ALANAGRIFFITH, MARIARODRIGUEZ, LOGANHILLMAN
    """
    upper = normalize_line(s).upper()

    if " " in upper:
        return False
    if school_pat.search(upper):
        return False
    if upper in single_word_non_names:
        return False
    if merged_name_like.fullmatch(upper) is None:
        return False

    # reject obvious all-caps title-ish short words
    if len(upper) <= 8:
        return False

    return True

def classify_line(line):
    if pd.isna(line):
        return np.nan

    s = normalize_line(line)
    if not s:
        return np.nan

    # age only
    if single_age.fullmatch(s):
        return "age_only"

    # school/company
    if school_pat.search(s):
        return "school_company"

    # obvious title / not-a-name
    if looks_like_title_not_name(s):
        return "dance_title"

    # explicit multiple people
    if any(p.fullmatch(s) for p in [two_and, two_age_paren, two_age_plain]):
        return "people"

    # comma-separated multiple people, but not LAST,FIRST or NAME,ST
    if "," in s and not name_rev.fullmatch(s) and not name_state.fullmatch(s):
        return "people"

    # single person
    if any(
        p.fullmatch(s)
        for p in [name_std, name_rev, name_state, name_age_dance, name_nickname, name_native_script]
    ):
        return "person"

    # probable merged name -> keep separate for review instead of losing it in other
    if maybe_manual_review_name(s):
        return "manual_review_name"

    return "other"

In [17]:
resolved = winners["Name"].apply(resolve_comma_name)
winners["Name_resolved"] = resolved.str[0]
winners["comma_status"] = resolved.str[1]

In [18]:
winners["Name_classfiy"] = winners["Name"].apply(classify_line)

In [19]:
winners.Name_classfiy.value_counts()

Name_classfiy
person                39651
dance_title            2056
other                   772
manual_review_name      285
school_company          157
people                   32
age_only                 13
Name: count, dtype: int64

In [35]:
# for name in sorted(winners[winners["Name_classfiy"].isin(["dance_title", "school_company"])]["Name"].dropna().unique()):
#     print(name)

In [ ]:
# for name in sorted(winners[winners["Name_classfiy"].isin(["other", "manual_review_name"])]["Name"].dropna().unique()):
#     print(name)

- Misspellings/errors that need to be counted as a person:
    - ALANAGRIFFITH = ALANA GRIFFITH
    - ALTHEA JOHNSON - STAUB
    - ANDREAJONUSAS = ANDREA JONUSAS
    - ELISA D[CARPIO = ELISA D'CARPIO
    - GABRIELLEGIRARD = GABRIELLE GIRARD
    - LOGANHILLMAN = LOGAN HILLMAN
    - MAKAI LEWIS - REES
    - MARIARODRIGUEZ = MARIA RODRIGUEZ
    - PHILIP MARTIN‐NIELSON
    - VICTORIAHASK = VICTORIA HASK
    - WING YAN [IBBY] CHOW

- Everything else in "other" seem to be actual dances

In [21]:
winners.loc[
    (winners.Name.str.contains("ALANAGRIFFITH", na=False)),
    "Name"
] = "ALANA GRIFFITH"

winners.loc[
    (winners.Name.str.contains("ANDREAJONUSAS", na=False)),
    "Name"
] = "ANDREA JONUSAS"

winners.loc[
    (winners.Name.str.contains("ELISA D[CARPIO", regex=False, na=False)),
    "Name"
] = "ELISA D'CARPIO"

winners.loc[
    (winners.Name.str.contains("GABRIELLEGIRARD", na=False)),
    "Name"
] = "GABRIELLE GIRARD"

winners.loc[
    (winners.Name.str.contains("LOGANHILLMAN", na=False)),
    "Name"
] = "LOGAN HILLMAN"

winners.loc[
    (winners.Name.str.contains("MARIARODRIGUEZ", na=False)),
    "Name"
] = "MARIA RODRIGUEZ"

winners.loc[
    (winners.Name.str.contains("VICTORIAHASK", na=False)),
    "Name"
] = "VICTORIA HASK"

winners.loc[
    (winners.Name.str.contains("ALANA GRIFFITH", na=False)) |
    (winners.Name.str.contains("ALTHEA JOHNSON - STAUB", na=False)) |
    (winners.Name.str.contains("ANDREA JONUSAS", na=False)) |
    (winners.Name.str.contains("ELISA D'CARPIO", na=False)) |
    (winners.Name.str.contains("GABRIELLE GIRARD", na=False)) |
    (winners.Name.str.contains("LOGAN HILLMAN", na=False)) |
    (winners.Name.str.contains("MARIA RODRIGUEZ", na=False)) |
    (winners.Name.str.contains("MAKAI LEWIS - REES", na=False)) |
    (winners.Name.str.contains("PHILIP MARTIN‐NIELSON", na=False)) |
    (winners.Name.str.contains("VICTORIA HASK", na=False)) |
    (winners.Name.str.contains("WING YAN [IBBY] CHOW", na=False)), 
    "Name_classfiy"
] = "person"

In [22]:
winners.loc[winners["Name_classfiy"].isin(["other", "manual_review_name"]), "Name_classfiy"] = "dance_title"

In [23]:
winners[winners.Name_classfiy == "age_only"]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category,Name_resolved,comma_status,Name_classfiy
31524,(15),MOSZKOVSKY WALTZ,2020,"CATTOLICA, ITALY",1ST PLACE,NaN,NaN,MOSZKOVSKY WALTZ,NaN,PAS DE DEUX,(15),not_comma,age_only
31527,(14),HARLEQUINADE,2020,"CATTOLICA, ITALY",2ND PLACE,NaN,NaN,HARLEQUINADE,NaN,PAS DE DEUX,(14),not_comma,age_only
31530,(18),SATANELLA,2020,"CATTOLICA, ITALY",3RD PLACE,NaN,NaN,SATANELLA,NaN,PAS DE DEUX,(18),not_comma,age_only
31533,(14),"ENT´ARTES - ESCOLA DE DANÇA, PORTUGAL",2020,"CATTOLICA, ITALY",TOP 6,NaN,NaN,"ENT´ARTES - ESCOLA DE DANÇA, PORTUGAL",NaN,PAS DE DEUX,(14),not_comma,age_only
31536,(14),"PROFESSIONE DANZA PARMA, ITALY",2020,"CATTOLICA, ITALY",TOP 6,NaN,NaN,"PROFESSIONE DANZA PARMA, ITALY",NaN,PAS DE DEUX,(14),not_comma,age_only
31539,(18),"IL BALLETTO, ITALY",2020,"CATTOLICA, ITALY",TOP 6,NaN,NaN,"IL BALLETTO, ITALY",NaN,PAS DE DEUX,(18),not_comma,age_only
31542,(13),"INDEPENDENT, ROMANIA",2020,"CATTOLICA, ITALY",TOP 6,NaN,NaN,"INDEPENDENT, ROMANIA",NaN,PAS DE DEUX,(13),not_comma,age_only
31545,(15),"IL BALLETTO, ITALY",2020,"CATTOLICA, ITALY",TOP 6,NaN,NaN,"IL BALLETTO, ITALY",NaN,PAS DE DEUX,(15),not_comma,age_only
31548,(16),"IL BALLETTO, ITALY",2020,"CATTOLICA, ITALY",TOP 6,NaN,NaN,"IL BALLETTO, ITALY",NaN,PAS DE DEUX,(16),not_comma,age_only
31691,(14),CONSERVATÓRIO INTERNACIONAL DE BALLET E DANÇA ...,2020,"BARCELONA, SPAIN",1ST PLACE,NaN,NaN,CONSERVATÓRIO INTERNACIONAL DE BALLET E DANÇA ...,NaN,PAS DE DEUX,(14),not_comma,age_only


- Potentially keep these for school awards, since people belong to certain schools (even w/o knowing their name)

In [24]:
winners.Name_classfiy.value_counts()

Name_classfiy
person            39662
dance_title        3102
school_company      157
people               32
age_only             13
Name: count, dtype: int64

In [ ]:
for name in sorted(
    winners[
        (winners.Name.str.contains(",", na=False)) & 
        ((winners.Name_classfiy == "people") |
        (winners.Name_classfiy == "person")) 
    ]["Name"].unique()
):
    print(name)

['ABIGAIL, BOIVIN',
 'ALESSANDRO FROLA, 12, MARTINA MANTOVANI, 13',
 'ALEXANDRIA, MARX',
 'APRIL RAE, MD',
 'BILLS, BILLS, BILLS',
 'BRITTANY, GEORGHEGAN',
 'CAROLIE, LAUVETZ',
 'CHANGING NOTHING, NOTHING CHANGES',
 'CHEN LI-EN, LIESL',
 'CONCERTO GROSSO, 4TH MOVEMENT',
 'CUBA LINDA, CUBA HERMOSA...',
 'DANCING ARTS CENTER, MA',
 'ELITE CLASSICAL COACHING, TX',
 'GENEVIEVE, COOVREY',
 'GENTLY FALLING, FLOATING DOWN',
 'GOH JIA YI, ANU',
 'JASMINE, JIMISON',
 'JESLYN, CHEN',
 'JESSICA ODASZ, ANDREA ASTUTO',
 'JESU, JOY!',
 'KATIA GARZA, TODD ERIC ALLEN',
 'KENNETH DUANE SHELBY, JR',
 'KENNETH SHELBY, JR',
 'KINSEY NOVAK, 13',
 'KORY GLATMAN, NJ',
 'LAUREN, LEB',
 'LI-EN, LIESL CHEN',
 'LOOKING BACK, GOING FORWARDS',
 'MASUMI, YOSHIMOTO',
 'MEGAN CARLO, FL',
 'MELISSA LORRY, MA',
 'MIRTH, FIRST MOVEMENT',
 'NADIA, TENCER',
 'NORA CLEMENTE, 16, BRYCE LEE, 16',
 'NY, NY',
 'PERHAPS, PERHAPS, PERHAPS',
 'PHYLICIA, ZI YING CHEW',
 'POUR TOI, MARCEAU',
 'RACHEL, ROHRICH',
 'RITA STROM, CA',
 

In [ ]:
for name in sorted(
    winners[
        (winners.Name.str.contains(",", na=False)) & 
        ((winners.Name_classfiy == "people") |
        (winners.Name_classfiy == "person")) 
    ]["Name_resolved"].unique()
):
    print(name)

['ABIGAIL, BOIVIN',
 'ALESSANDRO FROLA, 12, MARTINA MANTOVANI, 13',
 'ALEXANDRIA MARX',
 'APRIL RAE, MD',
 'BILLS, BILLS, BILLS',
 'BRITTANY, GEORGHEGAN',
 'CAROLIE, LAUVETZ',
 'CHANGING NOTHING, NOTHING CHANGES',
 'CHEN LI-EN, LIESL',
 'CONCERTO GROSSO, 4TH MOVEMENT',
 'CUBA LINDA, CUBA HERMOSA...',
 'DANCING ARTS CENTER, MA',
 'ELITE CLASSICAL COACHING, TX',
 'GENEVIEVE COOVREY',
 'GENTLY FALLING, FLOATING DOWN',
 'GOH JIA YI, ANU',
 'JASMINE JIMISON',
 'JESLYN CHEN',
 'JESSICA ODASZ, ANDREA ASTUTO',
 'JESU, JOY!',
 'KATIA GARZA, TODD ERIC ALLEN',
 'KENNETH DUANE SHELBY, JR',
 'KENNETH SHELBY, JR',
 'KINSEY NOVAK, 13',
 'KORY GLATMAN, NJ',
 'LAUREN LEB',
 'LI-EN, LIESL CHEN',
 'LOOKING BACK, GOING FORWARDS',
 'MASUMI YOSHIMOTO',
 'MEGAN CARLO, FL',
 'MELISSA LORRY, MA',
 'MIRTH, FIRST MOVEMENT',
 'NADIA, TENCER',
 'NORA CLEMENTE, 16, BRYCE LEE, 16',
 'NY, NY',
 'PERHAPS, PERHAPS, PERHAPS',
 'PHYLICIA, ZI YING CHEW',
 'POUR TOI, MARCEAU',
 'RACHEL ROHRICH',
 'RITA STROM, CA',
 'ROCHES

In [27]:
# for name in sorted(winners[
#     (winners.Name_classfiy == "person") |
#     (winners.Name_classfiy == "people")
# ]["Name"].unique()):
#     print(name)

- Dancers:
    - classifications: person, people

In [28]:
dancers = winners[
    (winners.Name_classfiy == "people") | 
    (winners.Name_classfiy == "person")
].copy()

mask = (dancers["Name_classfiy"] == "people") | (dancers.Name_classfiy == "person")
dancers.loc[mask, "Name"] = dancers.loc[mask, "Name"].str.split(",")
dancers = dancers.explode("Name")
dancers["Name"] = dancers["Name"].str.strip()

dancers

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category,Name_resolved,comma_status,Name_classfiy
6,KORY GLATMAN,NaN,2000,"WASHINGTON, DC",3RD PLACE,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,"KORY GLATMAN, NJ",ambiguous,person
6,NJ,NaN,2000,"WASHINGTON, DC",3RD PLACE,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,"KORY GLATMAN, NJ",ambiguous,person
7,ASHTON BERKLEY,"THE KIROV ACADEMY, DC",2000,"WASHINGTON, DC",2ND PLACE,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,ASHTON BERKLEY,not_comma,person
8,DANA GAULE,"ALLEGRO DANCE ARTS ACADEMY, NJ, USA",2000,"WASHINGTON, DC",1ST PLACE,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,DANA GAULE,not_comma,person
9,KATRINA DALTON,"ALLEGRO DANCE ARTS ACADEMY, NJ, USA",2000,"WASHINGTON, DC",HOPE AWARD,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,KATRINA DALTON,not_comma,person
...,...,...,...,...,...,...,...,...,...,...,...,...,...
43514,RIPPLE EFFECT,"PAS DE DEUX HAWAII, HI",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"PAS DE DEUX HAWAII, HI",LARGE ENSEMBLES,NaN,RIPPLE EFFECT,not_comma,person
43515,JULIET'S FRIENDS,"CITY BALLET SAN FRANCISCO, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"CITY BALLET SAN FRANCISCO, CA",LARGE ENSEMBLES,NaN,JULIET'S FRIENDS,not_comma,person
43518,GERDA'S RIVER JOURNEY,"BAYER BALLET ACADEMY, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"BAYER BALLET ACADEMY, CA",LARGE ENSEMBLES,NaN,GERDA'S RIVER JOURNEY,not_comma,person
43521,KIRA FARGAS,FOR ALL THE WORK,2026,"SAN FRANCISCO, CA (MAR)",OUTSTANDING CHOREOGRAPHER AWARD,NaN,NaN,FOR ALL THE WORK,SPECIAL AWARDS,NaN,KIRA FARGAS,not_comma,person


In [32]:
# for name in sorted(dancers.Name.unique()):
#     print(name)